[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabin2004/Machine-Learning-Bootcamp/blob/main/Module_08_Neural_Networks/01_neural_networks_intro.ipynb)

# Episode 20 – Introduction to Neural Networks

**Machine Learning Bootcamp** | Module 08

---

## 🎯 Learning Objectives
- Understand neurons, layers, and activation functions
- Implement forward propagation from scratch
- Grasp the intuition behind backpropagation and gradient descent

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

## 1. The Artificial Neuron

```
Inputs: x₁, x₂, ..., xₙ
Weights: w₁, w₂, ..., wₙ
Bias: b

z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b  (linear combination)
a = f(z)                              (activation function)
```

In [ ]:
# Common activation functions
z = np.linspace(-5, 5, 200)

activations = {
    'Sigmoid':  lambda x: 1 / (1 + np.exp(-x)),
    'Tanh':     np.tanh,
    'ReLU':     lambda x: np.maximum(0, x),
    'Leaky ReLU': lambda x: np.where(x > 0, x, 0.1 * x),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, fn) in zip(axes, activations.items()):
    ax.plot(z, fn(z), color='steelblue', lw=2)
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    ax.set_title(name)

plt.tight_layout(); plt.show()

## 2. Forward Propagation (2-Layer Network)

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

# Network: 2 inputs → 4 hidden neurons → 1 output
np.random.seed(42)
W1 = np.random.randn(4, 2) * 0.1   # (hidden_size, input_size)
b1 = np.zeros((4, 1))
W2 = np.random.randn(1, 4) * 0.1   # (output_size, hidden_size)
b2 = np.zeros((1, 1))

def forward(X):
    """X shape: (n_features, n_samples)"""
    Z1 = W1 @ X + b1       # linear
    A1 = relu(Z1)           # hidden activation
    Z2 = W2 @ A1 + b2      # linear
    A2 = sigmoid(Z2)        # output activation (probability)
    return A2

# Test on 3 samples
X_test = np.random.randn(2, 3)
output = forward(X_test)
print('Input  shape:', X_test.shape)
print('Output shape:', output.shape)
print('Predictions (probabilities):', output.flatten().round(4))

## 3. Backpropagation (Conceptual)

**Goal:** adjust weights to minimise the loss function.

$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial A} \cdot \frac{\partial A}{\partial Z} \cdot \frac{\partial Z}{\partial W}$$

Using the **chain rule** to propagate gradients backwards through the network.

Modern frameworks (TensorFlow, PyTorch) handle this automatically via **automatic differentiation**.

## 4. Training a Network from Scratch (XOR Problem)

In [ ]:
# XOR – not linearly separable, requires a hidden layer
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]]).T  # (2, 4)
y_xor = np.array([[0, 1, 1, 0]])               # (1, 4)

np.random.seed(0)
W1 = np.random.randn(4, 2)
b1 = np.zeros((4, 1))
W2 = np.random.randn(1, 4)
b2 = np.zeros((1, 1))
lr = 0.1

losses = []
for epoch in range(10000):
    # Forward
    Z1 = W1 @ X_xor + b1; A1 = np.tanh(Z1)
    Z2 = W2 @ A1 + b2;    A2 = sigmoid(Z2)
    loss = -np.mean(y_xor * np.log(A2 + 1e-9) + (1 - y_xor) * np.log(1 - A2 + 1e-9))
    losses.append(loss)

    # Backward
    m = X_xor.shape[1]
    dZ2 = A2 - y_xor
    dW2 = dZ2 @ A1.T / m;  db2 = dZ2.mean(axis=1, keepdims=True)
    dA1 = W2.T @ dZ2
    dZ1 = dA1 * (1 - A1 ** 2)  # tanh derivative
    dW1 = dZ1 @ X_xor.T / m;   db1 = dZ1.mean(axis=1, keepdims=True)

    W1 -= lr * dW1; b1 -= lr * db1
    W2 -= lr * dW2; b2 -= lr * db2

print('Final predictions:', (A2 > 0.5).astype(int).flatten(), '(expected: [0 1 1 0])')

plt.figure(figsize=(8, 4))
plt.plot(losses, color='steelblue')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('XOR Network – Training Loss')
plt.tight_layout(); plt.show()

## 🏋️ Exercises

1. Change the hidden activation from `tanh` to `relu`. Does the network still converge?
2. Add a second hidden layer with 4 neurons. Does it train faster?
3. Plot the decision boundary learned by the XOR network.

---
**Next ▶ [Episode 21 – Keras Basics](02_keras_basics.ipynb)**